# Exploração · `bronze.ocorrencias_rede`

Notebook de **investigação**, fora do pipeline. Descobre os problemas de qualidade das ocorrências de rede (interrupções de fornecimento) antes de escrever a Silver.

**Por que esta tabela importa:** ela é a base do **DEC e FEC por alimentador** (pedido da Beatriz e requisito 5.1) e da informação "a rua dele teve falta de luz" na visão 360. O glossário do case diz que **cada registro é uma UC interrompida**, então:
- DEC (horas) = soma das durações ÷ 60 ÷ número de UCs do alimentador;
- FEC = número de ocorrências ÷ número de UCs do alimentador.

Um erro de data ou de duração muda o DEC diretamente. Por isso as seções 9 e 10 são as mais importantes.


| # | Seção | Pergunta |
|---|---|---|
| 1 | Visão geral | Colunas, tipos e volume (o case fala em ~180 mil) |
| 2 | Nomes das colunas | Conferir os nomes reais antes de qualquer análise |
| 3 | Cabeçalho repetido | Há linha com o nome das colunas como dado? |
| 4 | Nulos e códigos de ausência | `N/A`, `NULL`, `-1`, `-`, vazio, `9999-12-31` |
| 5 | Duplicatas | Cópias exatas? Mesmo id com conteúdo diferente? |
| 6 | Formatos | Padrão de cada coluna |
| 7 | Categorias | Grafias diferentes de tipo, causa, status |
| 8 | Integridade | Órfãs? UCs de teste? Alimentador da ocorrência = alimentador da UC? |
| 9 | Datas de início e fim | Formatos, ordem dia/mês, fim antes do início |
| 10 | Duração | Números válidos? Bate com fim − início? Valores extremos? |
| 11 | Ocorrências em aberto | Sem data de fim: quantas e de quando? |
| 12 | Prévia do DEC/FEC | Valores por alimentador e mês, e os piores |
| 13 | Resumo | Regras propostas para a Silver |

**Pré-requisito:** `silver.unidades_consumidoras` gravada (seções 8 e 12).

In [0]:
from pyspark.sql import functions as F

catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

oc_raw = spark.table(f"{bronze_schema}.ocorrencias_rede")

def n_br(n):
    return f"{n:,}".replace(",", ".")

def sem_acento(col):
    """upper + trim + sem acento + espaços simples: para comparar grafias."""
    c = F.translate(F.upper(F.trim(col)), "ÁÀÂÃÉÊÍÓÔÕÚÜÇ", "AAAAEEIOOOUUC")
    return F.regexp_replace(c, r"\s+", " ")

def padrao(col):
    """Troca dígitos por 9 e letras por A: mostra o formato do valor."""
    return F.regexp_replace(F.regexp_replace(col, "[0-9]", "9"), "[A-Za-zÀ-ÿ]", "A")

## 1. Visão geral

**Achado:** 180.000 linhas, exatamente o volume do case. Oito colunas de origem, todas texto. O início da interrupção se chama `data_abertura` (e não `data_inicio`).

In [0]:
oc_raw.printSchema()
print(f"{n_br(oc_raw.count())} linhas (o case fala em ~180 mil)")
display(oc_raw.limit(10))

## 2. Nomes das colunas

**Esta é a única célula que depende dos nomes das colunas.** Compare com o `printSchema` da seção 1 e **corrija aqui** o que estiver diferente. Se uma coluna **não existir** na tabela (por exemplo, não houver duração pronta), troque o nome por `None`: as análises que dependem dela são puladas.

A célula para com erro se um nome preenchido não existir.

In [0]:
COL_ID = "id_ocorrencia"            # identificador da ocorrência
COL_UC = "id_uc"                    # UC interrompida
COL_ALIM = "alimentador"            # alimentador (None se não existir)
COL_INICIO = "data_abertura"        # início da interrupção (abertura da ocorrência)
COL_FIM = "data_encerramento"       # fim da interrupção
COL_DUR = "duracao_min"             # duração informada, em minutos (None se não existir)
COL_TIPO = "tipo"                   # ex.: falta de energia, fio partido (None se não existir)
COL_CAUSA = "causa"                 # ex.: vegetação, falha de equipamento (None se não existir)

preenchidas = [c for c in [COL_ID, COL_UC, COL_ALIM, COL_INICIO, COL_FIM, COL_DUR, COL_TIPO, COL_CAUSA] if c]
faltando = [c for c in preenchidas if c not in oc_raw.columns]
if faltando:
    raise ValueError(f"Colunas não encontradas: {faltando}. Colunas reais: {oc_raw.columns}. Corrija os nomes nesta célula.")

cols_origem = [c for c in oc_raw.columns if c not in ("source_file", "ingestion_timestamp")]
print("Colunas confirmadas:", preenchidas)
print("Todas as colunas de origem:", cols_origem)

## 3. Cabeçalho repetido

**O que explora:** linhas em que o valor da coluna é o próprio nome da coluna. A partir daqui, as análises usam `oc` (sem essas linhas).

**Achado:** nenhuma linha de cabeçalho repetido.

In [0]:
cab = oc_raw.filter(F.col(COL_ID) == COL_ID)
print("linhas de cabeçalho:", cab.count())
oc = oc_raw.filter(~F.coalesce(F.col(COL_ID) == COL_ID, F.lit(False)))
print("linhas sem cabeçalho:", n_br(oc.count()))

## 4. Nulos e códigos de ausência

**O que explora:** nulos reais e as formas de "vazio" das outras tabelas, incluindo a data coringa `9999-12-31` (em medidores e faturas, ela significava "ainda não aconteceu").

**Achado:** só a coluna `causa` tem ausência, em **4 formas**: nulo real (1.356), `-` (1.363), `N/A` (1.341) e `NULL` (1.340). Total: **5.400 linhas (3,0%)**. Nenhuma sentinela `9999-12-31`.

In [0]:
codigos = ["N/A", "NULL", "-1", "-", "", "9999-12-31"]
linhas = []
for c in cols_origem:
    linha = {"coluna": c, "nulo_real": oc.filter(F.col(c).isNull()).count()}
    for cod in codigos:
        linha[f"'{cod}'"] = oc.filter(F.trim(F.col(c)) == cod).count()
    linha["comeca_com_9999-12-31"] = oc.filter(F.trim(F.col(c)).startswith("9999-12-31")).count()
    linhas.append(linha)
display(spark.createDataFrame(linhas))

## 5. Duplicatas

**O que explora:**
- **Cópias exatas** (linhas idênticas): removidas sem perda, como em medidores e leituras.
- **Mesmo `id_ocorrencia` com conteúdo diferente:** mostra quais colunas mudam.
- **Mesma UC com o mesmo início:** a mesma interrupção registrada duas vezes com ids diferentes. Isso **dobraria o DEC e o FEC** daquela UC.

**Achado:** sem cópias (180.000 linhas e 180.000 ids distintos), nenhum id com versões diferentes e **nenhuma interrupção registrada em dobro** (0 pares de UC e início repetidos).

In [0]:
n = oc.count()
print("linhas:                          ", n_br(n))
print("linhas distintas (sem auditoria):", n_br(oc.select(cols_origem).distinct().count()))
print(f"{COL_ID} distintos:            ", n_br(oc.select(COL_ID).distinct().count()))
display(oc.groupBy(COL_ID).count().filter("count > 1")
        .groupBy("count").agg(F.count("*").alias("qtd_ids")).orderBy("count"))

versoes = oc.select(cols_origem).distinct()
ids_multi = versoes.groupBy(COL_ID).count().filter("count > 1").select(COL_ID)
print("ids com versões diferentes:", n_br(ids_multi.count()))
dif = versoes.join(ids_multi, COL_ID)
display(dif.groupBy(COL_ID).agg(
    *[(F.countDistinct(F.coalesce(F.col(c), F.lit("<nulo>"))) > 1).cast("int").alias(c)
      for c in cols_origem if c != COL_ID]
).select([F.sum(c).alias(c) for c in cols_origem if c != COL_ID]))
display(dif.orderBy(COL_ID).limit(10))

In [0]:
# Mesma UC e mesmo início, com ids diferentes (possível registro em dobro)
sem_copias = oc.dropDuplicates([COL_ID])
dup_evento = sem_copias.groupBy(COL_UC, COL_INICIO).count().filter("count > 1")
print("pares (UC, início) repetidos com ids diferentes:", n_br(dup_evento.count()))
display(sem_copias.join(dup_evento.select(COL_UC, COL_INICIO), [COL_UC, COL_INICIO])
        .orderBy(COL_UC, COL_INICIO).limit(20))

## 6. Formatos

**O que explora:** o padrão de cada coluna (dígitos viram `9`, letras viram `A`). Mostra ids fora do padrão, datas com e sem horário em formatos mistos, durações com vírgula ou texto.

**Achado:** ids, alimentador e `id_uc` com padrão único. **As duas datas vêm sempre em ISO com horário** (`yyyy-MM-dd HH:mm:ss`). O único problema de formato é em `duracao_min`: **16.200 valores (9,0%) têm o texto ` min` junto do número** (`28 min`, `127 min`).

In [0]:
for c in cols_origem:
    print(c)
    display(oc.groupBy(padrao(F.col(c)).alias("padrao")).count().orderBy(F.desc("count")).limit(12))

## 7. Categorias

**O que explora:** colunas com até 50 valores distintos, como vieram e depois de padronizar. Se o número cai, há grafias diferentes para o mesmo valor. A lista de **causas** é importante: ela define o que conta como "evitável por manutenção" no DEC/FEC (pendência do contrato da Gold).

**Achado:** `tipo` tem 5 valores limpos: `FALTA DE ENERGIA` (121.593), `DANO EM EQUIPAMENTO` (14.868), `OSCILACAO` (14.562), `SOBRECARGA` (14.547) e `FIO PARTIDO` (14.430). `causa` tem 6 valores reais (`CLIMATICA`, `SOBRECARGA`, `NAO IDENTIFICADA`, `VEGETACAO`, `TERCEIROS`, `FALHA DE EQUIPAMENTO`), além dos 4 códigos de ausência da seção 4. Atenção: `NAO IDENTIFICADA` é uma causa registrada (investigada sem conclusão), e não um dado faltando.

In [0]:
for c in cols_origem:
    brutos = oc.select(c).distinct().count()
    if brutos <= 50:
        padronizados = oc.select(sem_acento(F.col(c))).distinct().count()
        print(f"{c}: {brutos} valores brutos -> {padronizados} depois de padronizar")
        display(oc.groupBy(c).count().orderBy(F.desc("count")))

## 8. Integridade

**O que explora:**
- `id_uc` no padrão `UC` + 6 dígitos e existente na `silver.unidades_consumidoras`.
- Se alguma das **200 UCs de teste** aparece aqui.
- Se o **alimentador da ocorrência é o mesmo da UC** no cadastro. Se não for, o DEC/FEC por alimentador depende de qual dos dois a gente usa.

**Achado:** nenhuma órfã, nenhuma UC de teste e **o alimentador da ocorrência é sempre o mesmo do cadastro da UC**. Não há conflito sobre qual alimentador usar no DEC/FEC.

In [0]:
ucs = spark.table(f"{silver_schema}.unidades_consumidoras").select("id_uc", F.col("alimentador").alias("alim_cadastro"), "municipio")
ucs_bronze = (spark.table(f"{bronze_schema}.unidades_consumidoras")
              .filter(F.col("id_uc") != "id_uc").select("id_uc").distinct())
ucs_teste = ucs_bronze.join(ucs.select("id_uc"), "id_uc", "left_anti")

o = sem_copias.withColumnRenamed(COL_UC, "id_uc")
print("id_uc fora do padrão UC + 6 dígitos:",
      n_br(o.filter(~F.coalesce(F.col("id_uc").rlike("^UC[0-9]{6}$"), F.lit(False))).count()))
print("id_uc inexistente na Silver de UCs:  ", n_br(o.join(ucs, "id_uc", "left_anti").count()))
print("ocorrências em UCs de teste:         ", n_br(o.join(ucs_teste, "id_uc", "left_semi").count()))
display(o.join(ucs, "id_uc", "left_anti").groupBy(padrao(F.col("id_uc")).alias("padrao_id_uc")).count())

if COL_ALIM:
    comp = o.join(ucs, "id_uc").withColumn("alim_oc", F.upper(F.trim(F.col(COL_ALIM))))
    print("alimentador da ocorrência diferente do cadastro da UC:",
          n_br(comp.filter(F.col("alim_oc") != F.col("alim_cadastro")).count()))
    display(comp.filter(F.col("alim_oc") != F.col("alim_cadastro"))
            .select("id_uc", COL_ALIM, "alim_cadastro").limit(10))

## 9. Datas de início e fim

**O que explora:** formatos (com e sem horário) e a ordem dia/mês, com a mesma técnica das outras tabelas.
- **Teste 1, partes maiores que 12:** no hífen, a 2ª parte maior que 12 indica mês primeiro (convenção do projeto até agora).
- **Teste 2, gabarito pela duração:** se a tabela tem a duração em minutos, `fim − início` deve bater com ela. Com a ordem dia/mês certa, bate; com a trocada, não. É o mesmo tipo de gabarito do "vencimento = emissão + 12 dias" das faturas.
- **Datas impossíveis:** fim antes do início, datas fora de 2022 a 2026, sentinela.

**Achado:** só ISO com horário, então não há dúvida de ordem dia/mês (os dois testes dão o mesmo resultado). **Gabarito:** nas 163.800 linhas com duração numérica, `fim − início` bate com `duracao_min` em **100%**. Nenhuma data inválida, nenhum fim antes do início, tudo entre 2022 e jun/2026. Por ano: 2022: 20.168 (só o 2º semestre); 2023: 40.301; 2024: 40.355; **2025: 57.068 (+41%)**; 2026: 22.108 (só o 1º semestre).

In [0]:
datas = [c for c in [COL_INICIO, COL_FIM] if c]
for c in datas:
    print(c)
    display(oc.groupBy(padrao(F.trim(F.col(c))).alias("padrao")).count().orderBy(F.desc("count")))
    pt = (oc.withColumn("p", padrao(F.trim(F.col(c))))
            .filter(F.col("p").rlike("^99[-./]99[-./]9999"))
            .withColumn("sep", F.substring(F.trim(F.col(c)), 3, 1))
            .withColumn("p1", F.substring(F.trim(F.col(c)), 1, 2).cast("int"))
            .withColumn("p2", F.substring(F.trim(F.col(c)), 4, 2).cast("int")))
    display(pt.groupBy("sep").agg(F.count("*").alias("linhas"),
                                  F.sum((F.col("p1") > 12).cast("int")).alias("primeira_parte_maior_12"),
                                  F.sum((F.col("p2") > 12).cast("int")).alias("segunda_parte_maior_12")))

In [0]:
# Conversor por padrão: monta o formato a partir do padrão de cada valor
# ordem da data: 'mes_primeiro_no_hifen' = convenção do projeto (hífen = mês primeiro; ponto e barra = dia primeiro)
def formato_de(p, ordem="mes_primeiro_no_hifen"):
    data, resto = p[:10], p[10:]
    if data == "9999-99-99":
        fd = "yyyy-MM-dd"
    elif data in ("99-99-9999", "99.99.9999", "99/99/9999"):
        sep = data[2]
        dia_primeiro = (sep != "-") if ordem == "mes_primeiro_no_hifen" else (sep == "-")
        fd = f"dd{sep}MM{sep}yyyy" if dia_primeiro else f"MM{sep}dd{sep}yyyy"
    else:
        return None
    hora = {"": "", " 99:99": " HH:mm", " 99:99:99": " HH:mm:ss", "A99:99:99": "'T'HH:mm:ss", "A99:99": "'T'HH:mm"}.get(resto)
    return None if hora is None else fd + hora

def converte(col, ordem="mes_primeiro_no_hifen"):
    padroes = [r["p"] for r in oc.select(padrao(F.trim(F.col(col))).alias("p")).distinct().collect() if r["p"]]
    expr = None
    for p in padroes:
        fmt = formato_de(p, ordem)
        if fmt is None:
            continue
        cond = padrao(F.trim(F.col(col))) == p
        val = F.expr(f"try_to_timestamp(trim({col}), \"{fmt}\")")
        expr = F.when(cond, val) if expr is None else expr.when(cond, val)
    return expr if expr is not None else F.lit(None).cast("timestamp")

for c in datas:
    print(c, "| padrões sem formato conhecido:",
          [r["p"] for r in oc.select(padrao(F.trim(F.col(c))).alias("p")).distinct().collect()
           if r["p"] and formato_de(r["p"]) is None])

In [0]:
# Teste 2: gabarito pela duração (só se a tabela tiver a duração pronta)
if COL_DUR and COL_FIM:
    dur = F.expr(f"try_cast(regexp_replace(trim({COL_DUR}), ',', '.') as double)")
    for ordem in ["mes_primeiro_no_hifen", "dia_primeiro_no_hifen"]:
        t = (oc.withColumn("ini", converte(COL_INICIO, ordem)).withColumn("fim", converte(COL_FIM, ordem))
               .withColumn("dur", dur)
               .withColumn("dif_min", (F.unix_timestamp("fim") - F.unix_timestamp("ini")) / 60)
               .withColumn("sep_ini", F.when(padrao(F.trim(F.col(COL_INICIO))).rlike("^99[-./]99[-./]"), F.substring(F.trim(F.col(COL_INICIO)), 3, 1)).otherwise("ISO")))
        print(f"=== ordem: {ordem}")
        display(t.filter("ini IS NOT NULL AND fim IS NOT NULL AND dur IS NOT NULL").groupBy("sep_ini").agg(
            F.count("*").alias("linhas"),
            F.sum((F.abs(F.col("dif_min") - F.col("dur")) <= 1).cast("int")).alias("fim_menos_inicio_bate_com_duracao")))
else:
    print("Sem coluna de duração: o teste 2 não se aplica.")

In [0]:
# Com a convenção do projeto: datas não convertidas, fim antes do início, fora do período
# a sentinela 9999-12-31 vira nulo (como em medidores e faturas): significa "ainda sem fim"
fim_expr = (F.when(F.trim(F.col(COL_FIM)).startswith("9999-12-31"), F.lit(None).cast("timestamp")).otherwise(converte(COL_FIM))
            if COL_FIM else F.lit(None).cast("timestamp"))
oc_dt = o.withColumn("ini", converte(COL_INICIO)).withColumn("fim", fim_expr)
print("início não convertido:", n_br(oc_dt.filter(F.col(COL_INICIO).isNotNull() & F.col("ini").isNull()).count()))
if COL_FIM:
    print("fim não convertido (sem contar nulos e sentinela):",
          n_br(oc_dt.filter(F.col(COL_FIM).isNotNull() & ~F.trim(F.col(COL_FIM)).startswith("9999-12-31") & F.col("fim").isNull()).count()))
    print("fim antes do início:", n_br(oc_dt.filter("fim < ini").count()))
print("início fora de 2022-01-01 a 2026-06-30:",
      n_br(oc_dt.filter("ini < TIMESTAMP'2022-01-01' OR ini > TIMESTAMP'2026-06-30 23:59:59'").count()))
display(oc_dt.groupBy(F.year("ini").alias("ano")).count().orderBy("ano"))

## 10. Duração

**O que explora:** se a duração converte para número, valores zero, negativos e extremos (uma interrupção de vários dias numa UC é possível, mas rara), e a distribuição por causa. Se a tabela não tem a duração pronta, ela é calculada como `fim − início`.

**Achado:** as 16.200 durações que não convertem são todas as do sufixo ` min`. Nenhuma duração zero ou negativa; de 5 minutos a 48 horas (mediana 72 min; 244 acima de 24 h; nenhuma acima de 7 dias). **Causa climática domina:** 43.692 ocorrências com mediana de 101 min e **116.919 horas (36% do total)**; as outras causas ficam entre 65 e 67 min de mediana e ~39 mil horas cada.

In [0]:
if COL_DUR:
    oc_dur = oc_dt.withColumn("dur", F.expr(f"try_cast(regexp_replace(trim({COL_DUR}), ',', '.') as double)"))
    print("duração que não converte:", n_br(oc_dur.filter(F.col(COL_DUR).isNotNull() & F.col("dur").isNull()).count()))
    display(oc_dur.filter(F.col(COL_DUR).isNotNull() & F.col("dur").isNull()).groupBy(COL_DUR).count().limit(15))
else:
    oc_dur = oc_dt.withColumn("dur", (F.unix_timestamp("fim") - F.unix_timestamp("ini")) / 60)

print("duração zero:     ", n_br(oc_dur.filter("dur = 0").count()))
print("duração negativa: ", n_br(oc_dur.filter("dur < 0").count()))
print("mais de 24 horas: ", n_br(oc_dur.filter("dur > 1440").count()))
print("mais de 7 dias:   ", n_br(oc_dur.filter("dur > 10080").count()))
display(oc_dur.select(F.min("dur"), F.expr("percentile_approx(dur, array(0.01, 0.25, 0.5, 0.75, 0.99))").alias("p1_p25_mediana_p75_p99"), F.max("dur")))
if COL_CAUSA:
    display(oc_dur.groupBy(sem_acento(F.col(COL_CAUSA)).alias("causa")).agg(
        F.count("*").alias("ocorrencias"), F.round(F.expr("percentile_approx(dur, 0.5)"), 0).alias("duracao_mediana_min"),
        F.round(F.sum("dur") / 60, 0).alias("horas_totais")).orderBy(F.desc("horas_totais")))

## 11. Ocorrências em aberto

**O que explora:** ocorrências sem data de fim (nulo ou sentinela). Se forem antigas, provavelmente são registros não encerrados no sistema, e não interrupções que ainda estão acontecendo. Isso decide se entram no DEC.

**Achado:** nenhuma ocorrência sem data de fim.

In [0]:
if COL_FIM:
    aberto = oc_dur.filter(F.col("fim").isNull())
    print("sem data de fim:", n_br(aberto.count()))
    display(aberto.groupBy(F.date_format("ini", "yyyy-MM").alias("mes_inicio")).count().orderBy(F.desc("mes_inicio")).limit(15))
    if COL_DUR:
        print("sem fim, mas com duração preenchida:", n_br(aberto.filter("dur IS NOT NULL").count()))

## 12. Prévia do DEC/FEC

**O que explora:** uma primeira conta de DEC e FEC por alimentador e mês, com a fórmula simplificada do glossário. O denominador é o número de UCs de cada alimentador na `silver.unidades_consumidoras` (45.000 no total). Serve para ver se os valores fazem sentido e quais alimentadores aparecem como os piores, **antes** de qualquer limpeza.

**Achado:** **122 alimentadores** (o case fala em mais de 120). DEC mensal mediano de 0,14 h e FEC de 0,09.

**Atenção ao denominador:** os 7 piores DEC dos últimos 12 meses são alimentadores **pequenos** (20 a 47 UCs) com ~340 interrupções no ano: DEC de 16 a 38 horas e FEC de 8 a 17. Alimentadores grandes (ALM-SAO-004, ALM-JAB-004, ALM-REC-008, de 863 a 2.285 UCs) aparecem logo depois, com DEC de ~14 h, mas concentram muito mais **horas × UCs afetadas**. Para "onde a manutenção dá mais retorno", a tela precisa mostrar o DEC **e** o total de horas interrompidas, e não só o ranking de DEC.

In [0]:
ucs_por_alim = ucs.groupBy("alim_cadastro").agg(F.count("*").alias("qtd_ucs"), F.first("municipio").alias("municipio"))
base = (oc_dur.join(ucs.select("id_uc", "alim_cadastro"), "id_uc")
              .withColumn("mes", F.date_format("ini", "yyyy-MM"))
              .filter("dur IS NOT NULL AND dur >= 0"))
decfec = (base.groupBy("alim_cadastro", "mes").agg(F.count("*").alias("interrupcoes"), F.sum("dur").alias("minutos"))
              .join(ucs_por_alim, "alim_cadastro")
              .withColumn("dec_horas", F.round(F.col("minutos") / 60 / F.col("qtd_ucs"), 3))
              .withColumn("fec", F.round(F.col("interrupcoes") / F.col("qtd_ucs"), 3)))
print("alimentadores:", ucs_por_alim.count(), "(o case fala em mais de 120)")
display(decfec.select(F.expr("percentile_approx(dec_horas, array(0.5, 0.9, 0.99))").alias("dec_mediana_p90_p99"),
                      F.expr("percentile_approx(fec, array(0.5, 0.9, 0.99))").alias("fec_mediana_p90_p99")))

anual = (base.filter("mes BETWEEN '2025-07' AND '2026-06'").groupBy("alim_cadastro")
             .agg(F.count("*").alias("interrupcoes"), F.sum("dur").alias("minutos"))
             .join(ucs_por_alim, "alim_cadastro")
             .withColumn("dec_12m_horas", F.round(F.col("minutos") / 60 / F.col("qtd_ucs"), 2))
             .withColumn("fec_12m", F.round(F.col("interrupcoes") / F.col("qtd_ucs"), 2)))
print("Piores alimentadores nos últimos 12 meses (DEC):")
display(anual.orderBy(F.desc("dec_12m_horas")).select("alim_cadastro", "municipio", "qtd_ucs", "interrupcoes", "dec_12m_horas", "fec_12m").limit(10))

## 13. Resumo e regras para a Silver

| Problema | Linhas | Regra | Log |
|---|---|---|---|
| `causa` ausente em 4 formas (nulo, `-`, `N/A`, `NULL`) | 5.400 (3,0%) | nulo real; `NAO IDENTIFICADA` é mantida como causa | O1 |
| `duracao_min` com o texto ` min` | 16.200 (9,0%) | remover o sufixo e converter para inteiro | O2 |
| Datas como texto | 180.000 | `timestamp` (ISO com horário) | O3 |

**Verificado e sem problema:** sem cabeçalho, cópias, registros em dobro, órfãs ou UCs de teste; alimentador sempre igual ao do cadastro; datas só em ISO, sem fim antes do início e sem ocorrência em aberto; `fim − início = duração` em 100% das linhas numéricas; `tipo` e `causa` sem variações de grafia.

**Para o contrato da Gold:**
- **A origem traz horário:** a `fato_interrupcao` guarda início e fim como data e hora.
- **Causas evitáveis por manutenção (proposta):** `VEGETACAO` (poda), `FALHA DE EQUIPAMENTO` e `SOBRECARGA` (reforço de rede). `CLIMATICA` e `TERCEIROS` não são evitáveis pela distribuidora; `NAO IDENTIFICADA` fica de fora.
- **DEC com denominador pequeno:** mostrar junto o número de UCs e as horas totais, para não priorizar só alimentadores minúsculos.

**Para o grupo e a banca:**
- 2025 teve **41% mais ocorrências** que 2023 e 2024.
- A causa climática responde por 36% das horas interrompidas.
- Decisão a combinar: o tipo `OSCILACAO` (14.562) entra no DEC/FEC? O glossário diz que cada registro é uma UC interrompida, então a proposta é manter, registrando a suposição no README.